In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
from torch.utils.data import Dataset
from PIL import Image
import os
import torch

class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform

        self.image_files = sorted(os.listdir(images_dir))
        self.mask_files = sorted(os.listdir(masks_dir))

    def __len__(self):
        return len(self.image_files)



    def __getitem__(self, idx):
       img_path = os.path.join(self.images_dir, self.image_files[idx])
       mask_path = os.path.join(self.masks_dir, self.mask_files[idx])

       image = Image.open(img_path).convert("RGB")
       mask = Image.open(mask_path)

    # Resize
       from torchvision.transforms import functional as F

       image = F.resize(image, (256, 256))
       mask = F.resize(mask, (256, 256), interpolation=Image.NEAREST)

       image = F.to_tensor(image)

       mask = torch.from_numpy(np.array(mask)).long()
       mask = remap_mask(mask)

       return image, mask



In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [ ]:
train_dataset = SUIMDataset(
    images_dir="/kaggle/input/q3-stage3-2026/dataset/images",
    masks_dir="/kaggle/input/q3-stage3-2026/dataset/masks",
    transform=image_transform
)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

In [ ]:
import matplotlib.pyplot as plt

images, masks = next(iter(train_loader))

plt.figure(figsize=(12, 4))
for i in range(4):
    plt.subplot(2, 4, i+1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.axis("off")

    plt.subplot(2, 4, i+5)
    plt.imshow(masks[i], cmap="tab10")
    plt.axis("off")

plt.show()


In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO

import segmentation_models_pytorch as smp
import torch

!pip install segmentation-models-pytorch

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

x, y = next(iter(train_loader))
x = x.to(device)

with torch.no_grad():
    out = model(x)

print(out.shape)


In [ ]:
from torch.utils.data import random_split, DataLoader

dataset_size = len(train_dataset)
val_size = int(0.2 * dataset_size)
train_size = dataset_size - val_size

train_dataset, val_dataset = random_split(
    train_dataset, [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True, num_workers=2
)

val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=2
)


In [ ]:
# TO DO
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#Training Loop
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)

# Valid Loop
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

    return running_loss / len(dataloader)

# Full Training
num_epochs = 4

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss = validate_one_epoch(
        model, val_loader, criterion, device
    )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )


In [ ]:
# TO DO
import matplotlib.pyplot as plt

num_epochs = 5
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss = validate_one_epoch(
        model, val_loader, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

# Plot loss
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# that should be overfit isn't it ? i'm sorry i do not have time

In [ ]:
# TO DO
import matplotlib.pyplot as plt
import torch

model.eval()

images, masks = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

num_samples = min(4, images.size(0))

plt.figure(figsize=(12, 9))

for i in range(num_samples):
    # Original Image
    plt.subplot(num_samples, 3, i * 3 + 1)
    plt.imshow(images[i].permute(1, 2, 0).cpu())
    plt.title("Input Image")
    plt.axis("off")

    # Ground Truth
    plt.subplot(num_samples, 3, i * 3 + 2)
    plt.imshow(masks[i].cpu(), cmap="tab10")
    plt.title("Ground Truth")
    plt.axis("off")

    # Prediction
    plt.subplot(num_samples, 3, i * 3 + 3)
    plt.imshow(preds[i].cpu(), cmap="tab10")
    plt.title("Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()
